# FinReason synthetic dataset quality check

This companion notebook loads the public package, displays bounded summaries and reruns the dependency-free validator. All records are synthetic.

## Setup

Run from the dataset folder in Jupyter or Google Colab. No third-party Python packages are required.

In [ ]:
from pathlib import Path
import csv
import json
from collections import Counter

ROOT = Path.cwd()
if not (ROOT / 'data' / 'metadata.json').exists():
    candidates = list(ROOT.rglob('finreason_synthetic_dataset/data/metadata.json'))
    if not candidates:
        raise FileNotFoundError('Open this notebook from the extracted FinReason dataset folder.')
    ROOT = candidates[0].parents[1]
DATA = ROOT / 'data'
metadata = json.loads((DATA / 'metadata.json').read_text(encoding='utf-8'))
metadata

## Data

Confirm the documented row counts and inspect the transaction label distribution.

In [ ]:
with (DATA / 'transactions.csv').open(newline='', encoding='utf-8') as handle:
    transactions = list(csv.DictReader(handle))

summary = {
    'rows': len(transactions),
    'splits': dict(Counter(row['split'] for row in transactions)),
    'anomaly_labels': dict(Counter(row['is_anomaly'] for row in transactions)),
    'laundering_labels': dict(Counter(row['is_laundering'] for row in transactions)),
}
summary

## Checks

Run the same integrity, ledger, temporal, foreign-key, rule-coverage and pattern-semantic checks used to certify the release.

In [ ]:
import sys
sys.path.insert(0, str(ROOT))
from validate_finreason_dataset import validate

quality = validate(DATA)
assert quality['status'] == 'PASS', [row for row in quality['checks'] if row['status'] == 'FAIL']
{'status': quality['status'], 'passed': quality['checks_passed'], 'failed': quality['checks_failed']}

## Results

Display scenario coverage without printing the full tables.

In [ ]:
with (DATA / 'scenarios.csv').open(newline='', encoding='utf-8') as handle:
    scenarios = list(csv.DictReader(handle))
scenario_counts = Counter(row['scenario_type'] for row in scenarios)
dict(sorted(scenario_counts.items()))

## Takeaways

A passing validator means the release is internally consistent and the planted scenario ground truth satisfies its declared structural constraints. It does not establish real-world detection performance. Report synthetic, IBM AML and PaySim results separately.